# 04 — RSA & Figures

This notebook:
1. Runs Representational Similarity Analysis (RSA) per ROI
2. Tests model RDMs (load-only, abstract-only, combined) against neural RDMs
3. Performs group-level inference with FDR correction
4. Generates publication-quality figures

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, ttest_1samp
from scipy.spatial.distance import squareform

sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))

sns.set_style('whitegrid')
sns.set_context('talk')
%matplotlib inline

## Configuration

In [ ]:
# ======================== EDIT THESE PATHS ========================
NSD_ROOT = '/path/to/nsd'                    # NSD data root (for ROI masks)
# =================================================================

SCORES_FILE = 'data/nsd_image_scores.csv'
BETA_DIR = 'results/beta_maps/'
RSA_DIR = 'results/rsa_results/'
SUBJECTS = [f'sub-{i:02d}' for i in range(1, 9)]
ROIS = ['nsdgeneral', 'V1v', 'V2v', 'V3v', 'hV4', 'LOC', 'OFA', 'PPA', 'RSC']
FDR_Q = 0.05

os.makedirs(RSA_DIR, exist_ok=True)
os.makedirs('results/group', exist_ok=True)

## Step 1: Compute Model RDMs

In [ ]:
from rsa import compute_model_rdm, compute_single_dim_rdm, compute_rdm_neural, rsa_correlation, fisher_z

df_scores = pd.read_csv(SCORES_FILE)
print(f'Loaded {len(df_scores)} scores')

## Step 2: RSA per Subject × ROI

In [ ]:
from rsa import get_roi_indices

model_names = ['combined', 'load_only', 'abstract_only']

# Results: {roi: {model: [r_sub1, r_sub2, ...]}}
results = {roi: {m: [] for m in model_names} for roi in ROIS}
subject_labels = []

for subject in SUBJECTS:
    beta_file = os.path.join(BETA_DIR, f'{subject}_betas.npz')
    if not os.path.exists(beta_file):
        print(f'  Skipping {subject}')
        continue

    data = np.load(beta_file)
    Y = data['betas']
    sub_nsd_ids = data['nsd_ids']
    n_voxels = Y.shape[1]

    # Align scores
    df_sub = df_scores[df_scores['nsd_id'].isin(sub_nsd_ids)].copy()
    df_sub = df_sub.set_index('nsd_id').loc[sub_nsd_ids].reset_index()
    valid_mask = df_sub['load_z'].notna() & df_sub['abstractness_z'].notna()
    df_sub = df_sub[valid_mask]
    Y = Y[valid_mask.values]

    # Model RDMs
    load_scores = df_sub['load_z'].values
    abstract_scores = df_sub['abstractness_z'].values
    combined_rdm = compute_model_rdm(np.column_stack([load_scores, abstract_scores]))
    load_rdm = compute_single_dim_rdm(load_scores)
    abstract_rdm = compute_single_dim_rdm(abstract_scores)
    model_rdms = {'combined': combined_rdm, 'load_only': load_rdm, 'abstract_only': abstract_rdm}

    subject_labels.append(subject)

    for roi_name in ROIS:
        roi_mask = get_roi_indices(NSD_ROOT, subject, roi_name, n_voxels)
        roi_betas = Y[:, roi_mask] if roi_mask is not None else Y

        if roi_betas.shape[1] == 0:
            for m in model_names:
                results[roi_name][m].append(np.nan)
            continue

        neural_rdm = compute_rdm_neural(roi_betas)
        for m in model_names:
            r = rsa_correlation(model_rdms[m], neural_rdm)
            results[roi_name][m].append(r)

    print(f'{subject}: done')

print(f'\nProcessed {len(subject_labels)} subjects')

## Step 3: Group-Level Inference

In [ ]:
from rsa import group_ttest_rsa, fdr_correction

group_records = []
for roi_name in ROIS:
    for model_name in model_names:
        r_values = [r for r in results[roi_name][model_name] if not np.isnan(r)]
        if len(r_values) < 2:
            continue
        mean_r, t_stat, p_val = group_ttest_rsa(r_values)
        group_records.append({
            'roi': roi_name, 'model': model_name,
            'mean_r': mean_r, 't_stat': t_stat, 'p_value': p_val,
            'n_subjects': len(r_values)
        })

df_group = pd.DataFrame(group_records)

# FDR correction
p_vals = df_group['p_value'].fillna(1.0).values
df_group['significant'] = fdr_correction(p_vals, q=FDR_Q)

print(df_group.to_string(index=False))

In [ ]:
# Save
df_group.to_csv(os.path.join(RSA_DIR, 'rsa_group_results.csv'), index=False)

# Per-subject r-values
per_sub = []
for roi in ROIS:
    for model in model_names:
        for i, r in enumerate(results[roi][model]):
            per_sub.append({'roi': roi, 'model': model,
                           'subject': subject_labels[i] if i < len(subject_labels) else f'sub-{i}',
                           'rsa_r': r, 'rsa_z': fisher_z(r) if not np.isnan(r) else np.nan})
df_per_sub = pd.DataFrame(per_sub)
df_per_sub.to_csv(os.path.join(RSA_DIR, 'rsa_per_subject.csv'), index=False)
print(f'Saved results to {RSA_DIR}')

## Figures

### Figure 1: RSA Bar Chart per ROI

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data for grouped bar plot
bar_width = 0.25
x = np.arange(len(ROIS))

colors = {'load_only': 'steelblue', 'abstract_only': 'mediumpurple', 'combined': 'teal'}
labels = {'load_only': 'Load', 'abstract_only': 'Abstractness', 'combined': 'Combined'}

for j, model in enumerate(model_names):
    means = []
    sems = []
    for roi in ROIS:
        vals = [r for r in results[roi][model] if not np.isnan(r)]
        means.append(np.mean(vals) if vals else 0)
        sems.append(np.std(vals) / np.sqrt(len(vals)) if len(vals) > 1 else 0)

    offset = (j - 1) * bar_width
    bars = ax.bar(x + offset, means, bar_width, yerr=sems,
                  label=labels[model], color=colors[model], capsize=3, alpha=0.85)

    # Mark significant bars
    for i, roi in enumerate(ROIS):
        row = df_group[(df_group['roi'] == roi) & (df_group['model'] == model)]
        if len(row) > 0 and row.iloc[0]['significant']:
            ax.text(x[i] + offset, means[i] + sems[i] + 0.005, '*',
                    ha='center', va='bottom', fontsize=14, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(ROIS, rotation=45, ha='right')
ax.set_ylabel('Mean RSA Correlation (Spearman r)')
ax.set_title('RSA: Model–Brain Similarity per ROI')
ax.axhline(0, color='gray', ls='--', lw=0.5)
ax.legend(title='Model RDM')

plt.tight_layout()
plt.savefig('results/group/rsa_bar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: results/group/rsa_bar_chart.png')

### Figure 2: RSA Heatmap

In [ ]:
# Heatmap: ROI × Model, colored by mean RSA r
pivot = df_group.pivot(index='roi', columns='model', values='mean_r')
pivot = pivot.reindex(index=ROIS, columns=model_names)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('RSA Correlation: ROI × Model RDM')
ax.set_ylabel('ROI')
ax.set_xlabel('Model RDM')

plt.tight_layout()
plt.savefig('results/group/rsa_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

### Figure 3: Model RDMs Visualization

In [ ]:
# Visualize model RDMs using scores from the selected stimuli
df_sel = pd.read_csv('data/selected_image_ids.csv')
valid = df_sel['load_z'].notna() & df_sel['abstractness_z'].notna()
df_sel = df_sel[valid].head(100)  # Use first 100 for visibility

load_scores = df_sel['load_z'].values
abstract_scores = df_sel['abstractness_z'].values

combined_rdm = compute_model_rdm(np.column_stack([load_scores, abstract_scores]))
load_rdm = compute_single_dim_rdm(load_scores)
abstract_rdm = compute_single_dim_rdm(abstract_scores)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, rdm, title in zip(axes,
                           [load_rdm, abstract_rdm, combined_rdm],
                           ['Load RDM', 'Abstractness RDM', 'Combined RDM']):
    im = ax.imshow(rdm, cmap='viridis', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('Image')
    ax.set_ylabel('Image')
    plt.colorbar(im, ax=ax, shrink=0.7, label='Distance')

plt.tight_layout()
plt.savefig('results/group/model_rdms.png', dpi=300, bbox_inches='tight')
plt.show()

### Figure 4: Per-Subject RSA Values (Swarm Plot)

In [ ]:
# Swarm plot of per-subject RSA values for key ROIs
key_rois = ['V1v', 'hV4', 'LOC', 'PPA', 'RSC']
df_plot = df_per_sub[df_per_sub['roi'].isin(key_rois) & df_per_sub['rsa_r'].notna()].copy()

fig, axes = plt.subplots(1, len(model_names), figsize=(6 * len(model_names), 5), sharey=True)

for ax, model in zip(axes, model_names):
    sub = df_plot[df_plot['model'] == model]
    sns.stripplot(data=sub, x='roi', y='rsa_r', order=key_rois,
                  jitter=True, alpha=0.7, size=8, ax=ax)
    sns.boxplot(data=sub, x='roi', y='rsa_r', order=key_rois,
                whis=1.5, showfliers=False, color='lightgray',
                boxprops=dict(alpha=0.4), ax=ax)
    ax.axhline(0, color='gray', ls='--', lw=0.5)
    ax.set_title(f'{model.replace("_", " ").title()} Model')
    ax.set_xlabel('ROI')
    if ax == axes[0]:
        ax.set_ylabel('RSA Correlation (r)')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Per-Subject RSA Values', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('results/group/rsa_swarm.png', dpi=300, bbox_inches='tight')
plt.show()

### Figure 5: Hypothesis Summary Table

In [ ]:
# Summarize hypothesis tests
print('=' * 80)
print('HYPOTHESIS EVALUATION SUMMARY')
print('=' * 80)

# H1: Load → V1-V4
print('\nH1: Stimulation load drives stronger activation in early visual cortex')
for roi in ['V1v', 'V2v', 'V3v', 'hV4']:
    row = df_group[(df_group['roi'] == roi) & (df_group['model'] == 'load_only')]
    if len(row) > 0:
        r = row.iloc[0]
        sig = '***' if r['significant'] else ''
        print(f'  {roi}: r={r["mean_r"]:.4f}, t={r["t_stat"]:.3f}, p={r["p_value"]:.4f} {sig}')

# H2: Abstractness → DMN
print('\nH2: Abstractness engages default mode regions')
for roi in ['nsdgeneral', 'LOC', 'PPA']:
    row = df_group[(df_group['roi'] == roi) & (df_group['model'] == 'abstract_only')]
    if len(row) > 0:
        r = row.iloc[0]
        sig = '***' if r['significant'] else ''
        print(f'  {roi}: r={r["mean_r"]:.4f}, t={r["t_stat"]:.3f}, p={r["p_value"]:.4f} {sig}')

# H3: Interaction → LOC, PPA, RSC
print('\nH3: Load × Abstractness interaction in LOC/PPA/RSC')
# Check GLM interaction term
glm_file = 'results/group/group_glm.npz'
if os.path.exists(glm_file):
    glm = np.load(glm_file, allow_pickle=True)
    reg_names = list(glm['regressor_names'])
    if 'load_x_abstract' in reg_names:
        idx = reg_names.index('load_x_abstract')
        sig_mask = glm['significant_mask']
        n_sig = sig_mask[idx].sum()
        print(f'  Interaction: {n_sig} significant voxels (whole brain)')

# RSA combined model for these ROIs
for roi in ['LOC', 'PPA', 'RSC']:
    row = df_group[(df_group['roi'] == roi) & (df_group['model'] == 'combined')]
    if len(row) > 0:
        r = row.iloc[0]
        sig = '***' if r['significant'] else ''
        print(f'  {roi} (combined RSA): r={r["mean_r"]:.4f}, t={r["t_stat"]:.3f}, p={r["p_value"]:.4f} {sig}')

print('\n' + '=' * 80)